# Calculate the mass fraction of Li-rich crust relative to the regular crust mass

I'm going to use the same sources for the mass fractions as :

- log($M_{CVZ}/M_*$), which is stored as log_q in my tables
- $M_*$
- log(H/He)
- Hydrogen atomic mass
- Helium atomic mass

In [1]:
from __future__ import print_function

#import matplotlib

#matplotlib.use('pdf')
#savefig=True
    
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
import periodictable as pt

start = time.time()
print(start)
time_string=str(start).split('.')[0]

#from mendeleev import O, Ca, Li, Na, Si, Fe, Mg, He
start = time.time()

#import wdatmos
import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs


#print(os.getcwd())

1724705044.4215748
all_avg
(116, 4, 27)
(4, 27, 116)
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov10_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov10_diffusion_timescales.csv


In [2]:
plt.show()

In [3]:
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [4]:
#wd_abund_file='20211112_all_wd_abundances_beryllium_objects_partially_added.csv'
#wd_abund_file= '20220303_all_wd_abundances_newMC_ages_allCa_abunds.csv'
wd_abund_file='20230207_WD_SSP_abundances_noJ2356.csv'

In [5]:
mLi_mine=0.007 #from Kesler et al. 2012
mCa_mine=0.0415 #assumed to be the same in all other elements as continental crust and Li enhancement to be negligible in mass fraction of others
mLi_crust=0.00002
mCa_crust=0.0415

In [6]:
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
wd_abund_table.add_index('name')

In [7]:
def get_mine_mass(row):
    numerator=(10**row['ssp_li/ca']/(pt.elements[20].mass/pt.elements[3].mass)*mCa_crust-mLi_crust)
    denominator=(mLi_mine-mLi_crust)-(10**row['ssp_li/ca']/(pt.elements[20].mass/pt.elements[3].mass)*(mCa_mine-mCa_crust))
    print('log(Li/Ca)',row['ssp_li/ca'])
    print('numerator',numerator)
    print('denominator',denominator)
    return numerator/denominator

In [8]:
def get_H_mass(row, H_hidden_R=0):
    denominator=1+(pt.elements[2].mass/pt.elements[1].mass)*10.**(-1*row['h/he'])
    H_mass= 10.**(row['log_q'])/denominator * row['m_wd']
    H_mass= H_mass*(1.+H_hidden_R) #Rolland et al. 2018 equation 4 for trace hydrogen to regain the hydrogen diffused below the convective envelope
    print(row['name'],'modeler:',row['modeler'],', Hydrogen mass in Convective Envelope:',H_mass, 'M_sol')
    return H_mass

In [9]:
for row in wd_abund_table:
    print('\n*******')
    print(row['name'])
    #H_hidden_R=2 #value from Rolland that he uses, admittedly at much higher temperatures than we're looking at here.
    H_hidden_R=0 #value assuming effectively all of the hydrogen is located in the convective envelope
    #get_H_mass(row, H_hidden_R=H_hidden_R)
    print(row['ssp_li/ca'],np.log10(mLi_mine/mCa_mine*(pt.elements[20].mass/pt.elements[3].mass)))
    print(pt.elements[20].mass,pt.elements[3].mass)
    #row['ssp_li/ca']=np.log10(mLi_crust/mCa_crust*(pt.elements[20].mass/pt.elements[3].mass))
    #row['ssp_li/ca']=np.log10(mLi_mine/mCa_mine*(pt.elements[20].mass/pt.elements[3].mass))
    print(row['ssp_li/ca'])
    mine_mass=get_mine_mass(row)
    print('mine mass fraction relative to continental crust:',mine_mass)
    print('*********\n')


*******
WDJ1644-0449
-2.25 -0.011466060163059523
40.078 6.941
-2.25
log(Li/Ca) -2.25
numerator 2.0417002403579974e-05
denominator 0.00698
mine mass fraction relative to continental crust: 0.002925071977590254
*********


*******
SDSSJ1330+6435
-2.08 -0.011466060163059523
40.078 6.941
-2.08
log(Li/Ca) -2.08
numerator 3.978112727974973e-05
denominator 0.00698
mine mass fraction relative to continental crust: 0.005699301902542942
*********


*******
WDJ1824+1213
-2.56 -0.011466060163059523
40.078 6.941
-2.56
log(Li/Ca) -2.56
numerator -2.0460828036295201e-07
denominator 0.00698
mine mass fraction relative to continental crust: -2.931350721532264e-05
*********


*******
WDJ2317+1830
-0.82 -0.011466060163059523
40.078 6.941
-0.82
log(Li/Ca) -0.82
numerator 0.001067837686396333
denominator 0.00698
mine mass fraction relative to continental crust: 0.15298534189059212
*********


*******
LHS2534
-1.93 -0.011466060163059523
40.078 6.941
-1.93
log(Li/Ca) -1.93
numerator 6.444308674251807e-05
de

Stuff below from page 17 of General Clemens XII

In [10]:
def get_el_mass(el,row):
    mass= row['m_wd']*10.**(row['log_q'])*10.**(row[el.lower()+'/he']-(3*row[el.lower()+'/he_err']))*(pt.elements[el_dict[el]].mass/pt.elements[2].mass)
    #that mass was in M_sol values
    mass=(mass*const.M_sun).to(u.kg)
    print(row['display_name'],row['modeler'])
    print(row['display_name'],mass)
    print(row['display_name'],mass.value*1e-3,'metric tons') 
    print(row['display_name'],mass.value*1e-3*1e-6,'Mega tons (Mt)')
    return

In [11]:
el_dict={
    'H':1,
    "Li":3,
}

In [12]:
for row in wd_abund_table:
    get_el_mass('Li',row)

KeyError: 'log_q'

In [ ]:
type(el_dict['Li'])

In [ ]:
const.M_earth.to(u.kg).value*1e-3*1e-6

In [ ]:
total_cont_crust_li=20.*1e-6*(4*1e25*u.g)
print(total_cont_crust_li.to(u.kg).value*1e-3*1e-6,'Mt (mega tons)')

In [ ]:
(4*1e25*u.g)/const.M_earth.to(u.g)

In [ ]:
(0.7e-2)/(20.e-6)